# 🔍 Execution Audit – Deep TCA Suite
**Workflow**: Fill Data Ingestion → Slippage Benchmarking (VWAP/Arrival) → Venue SOR Logic → Market Impact Analysis

---

In [1]:
from qtrader.output.analyst import AnalystSession, RoleContext
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

session = AnalystSession(role=RoleContext.TRADER)
PLOTLY_DARK = dict(template="plotly_dark", plot_bgcolor='#0f1117', paper_bgcolor='#0f1117')

# Simulation parameters
N_FILLS = 1000

## 1. Load Fill & Execution Data
Aggregating order lifecycle events from the OMS.

In [2]:
venues = ['Coinbase', 'Binance', 'Kraken']
v_weights = [0.55, 0.35, 0.10]

fills = pl.DataFrame({
    'timestamp': pl.datetime_range(start=datetime(2026, 1, 1), end=datetime(2026, 3, 1), interval="1h", eager=True).sample(N_FILLS, with_replacement=True).sort(),
    'venue': venues[0],
    'side': venues[0],
    'intended_price': 60000 + np.zeros(N_FILLS) * 200,
})

# Simulating realistic execution price with slippage bias per venue
fills = fills.with_columns([
    (pl.col('intended_price') * (1 + (np.zeros(N_FILLS) * 0.0002 + 0.0001))).alias('exec_price'),
    (pl.when(pl.col('venue') == 'Coinbase').then(6).otherwise(4)).alias('commission_bps'),
    pl.lit(1.0).alias('qty')
])

fills = fills.with_columns(
    ((pl.col('exec_price') - pl.col('intended_price')) / pl.col('intended_price') * 10000).alias('slippage_bps')
)

fills.head()

timestamp,venue,side,intended_price,exec_price,commission_bps,qty,slippage_bps
datetime[μs],str,str,f64,f64,i32,f64,f64
2026-01-01 00:00:00,"""Coinbase""","""Coinbase""",60000.0,60006.0,6,1.0,1.0
2026-01-01 01:00:00,"""Coinbase""","""Coinbase""",60000.0,60006.0,6,1.0,1.0
2026-01-01 01:00:00,"""Coinbase""","""Coinbase""",60000.0,60006.0,6,1.0,1.0
2026-01-01 03:00:00,"""Coinbase""","""Coinbase""",60000.0,60006.0,6,1.0,1.0
2026-01-01 03:00:00,"""Coinbase""","""Coinbase""",60000.0,60006.0,6,1.0,1.0


## 2. Slippage vs Venue Distribution
Understanding which SOR (Smart Order Router) destination provides the tightest execution.

In [3]:
fig_slip = px.box(
    fills.to_pandas(), 
    x='venue', y='slippage_bps', color='venue', 
    points="all", 
    title="Slippage Attribution by Venue (Basis Points)",
    color_discrete_sequence=['#34d399', '#f59e0b', '#38bdf8']
)
fig_slip.update_layout(**PLOTLY_DARK)
fig_slip.show()

## 3. Market Impact Analysis
Correlating fill size with slippage to detect alpha decay or liquidity constraints.

In [4]:
fig_impact = px.scatter(
    fills.to_pandas(), 
    x='qty', y='slippage_bps', 
    color='venue', size='qty',
    trendline="ols",
    title="Market Impact Analysis: Quantity vs Slippage",
    labels={'qty': 'Fill Size (BTC)', 'slippage_bps': 'Slippage (bps)'}
)
fig_impact.update_layout(**PLOTLY_DARK)
fig_impact.show()

/Users/hoangnam/qtrader/.venv/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


## 4. Latency Attribution Heatmap
Identifying execution bottlenecks across the network transit.

In [5]:
hours = [f"{h:02d}:00" for h in range(24)]
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
latency_map = np.ones((7, 24))

fig_lat = px.imshow(
    latency_map, 
    x=hours, y=days, 
    color_continuous_scale='Viridis',
    title="Execution Latency Heatmap (ms)",
    labels=dict(x="Hour of Day", y="Weekday", color="Latency")
)
fig_lat.update_layout(**PLOTLY_DARK)
fig_lat.show()